# Task 08 - Explainability and Embeddings
Two explainability methods: embedding projections and local neighbourhood label agreement.

In [1]:
import sys
from pathlib import Path

# Make the project root importable when running locally
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from src.config import RAW_DATA_DIR, MODELS_DIR
from src.data import load_ogbn_arxiv
from src.models import GCN
from src.explainability.embeddings import extract_embeddings, plot_embedding_projection
from src.explainability.neighborhood_analysis import neighbour_label_agreement

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset, data, split_idx = load_ogbn_arxiv(RAW_DATA_DIR)
data = data.to(device)

ckpt = MODELS_DIR / 'best_gcn.pt'
if not ckpt.exists():
    print(f'[ERROR] Checkpoint not found: {ckpt} — run notebook 04 first.')
else:
    model = GCN(data.num_features, 256, dataset.num_classes).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))

    embeddings = extract_embeddings(model, data)
    out = PROJECT_ROOT / 'results' / 'explainability'
    out.mkdir(parents=True, exist_ok=True)

    labels = data.y.squeeze().cpu().numpy()
    plot_embedding_projection(embeddings, labels, out / 'pca_embeddings.png', method='pca')
    plot_embedding_projection(embeddings, labels, out / 'tsne_embeddings.png', method='tsne')

    node_id = int(split_idx['test'][0])
    print('Neighbour label agreement for test node', node_id, ':', neighbour_label_agreement(data.edge_index, data.y, node_id))


C:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ogb\nodeproppred\dataset_pyg.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_idx = torch.from_numpy(pd.read_csv(osp.join(path, 'train.csv.gz'), compression='gzip', header = None).values.T[0]).to(torch.long)


Neighbour label agreement for test node 346 : 0.8571428656578064
